# Global Parameter Optimization: Cross-Subject Forward Model

Estimate an individualized forward model by optimizing global parameters
of the fsaverage template against sensor covariance from the MNE sample subject.

**Pipeline (11 parameters):**
1. Build ground truth forward model from the sample subject (our "unknown" individual)
2. Simulate EEG → sensor covariance (this is what we'd have from real data)
3. Stepwise optimization of fsaverage template via subspace alignment:
   - Stage 1: Skull conductivity (1p)
   - Stage 2: + brain offset (4p)
   - Stage 3: + per-axis stretch (7p)
   - **Stage 4: + rotation + co-deform + ratio-constrained σ (11p) — current best**
4. Build optimized forward model, invert test data
5. Compare: ground truth on sample brain vs inverse solution on optimized fsaverage

**Current best model:** Co-deform + rotation (11p) — MLE 20.6, EMD 28.8, cost 0.060

In [1]:
import os
import shutil

import mne
import numpy as np
from IPython.display import display, Image
from scipy.optimize import minimize as sp_minimize
from scipy.spatial.distance import cdist
from scipy.spatial.transform import Rotation

from invert import Solver
from invert.simulate import SimulationConfig, SimulationGenerator
from invert.forward.forward import _get_fsaverage_dir, get_info
from invert.cortical_folding.fast_bem import (
    make_bem_solution_factored,
    rebuild_bem_from_coeff,
    partial_K_update,
)
from invert.cortical_folding.cache import save_params, load_params
from mne.bem import _check_complete_surface

## 1. Setup: Ground Truth (Sample Subject)

In [2]:
# Paths
fs_dir = _get_fsaverage_dir(verbose=0)
subjects_dir = os.path.dirname(fs_dir)
trans_fs = os.path.join(fs_dir, 'bem', 'fsaverage-trans.fif')
info = get_info()

sample_path = mne.datasets.sample.data_path()
subjects_dir_sample = os.path.join(sample_path, 'subjects')

# Build ground truth forward model from the sample subject
print('Building ground truth forward model (sample subject)...')
fwd_true = mne.make_forward_solution(
    info,
    trans=os.path.join(sample_path, 'MEG', 'sample', 'sample_audvis_raw-trans.fif'),
    src=mne.setup_source_space('sample', spacing='ico3', surface='white',
        subjects_dir=subjects_dir_sample, add_dist=False, verbose=0),
    bem=os.path.join(subjects_dir_sample, 'sample', 'bem',
        'sample-5120-5120-5120-bem-sol.fif'),
    eeg=True, mindist=0.0, verbose=0,
)
fwd_true = mne.convert_forward_solution(
    fwd_true, surf_ori=True, force_fixed=True, verbose=0,
)
pos_true = fwd_true['source_rr']
n_dipoles = fwd_true['sol']['data'].shape[1]
print(f'Ground truth: {n_dipoles} dipoles')

/Users/lukas/projects/invert/invert-package/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building ground truth forward model (sample subject)...
Ground truth: 1282 dipoles


## 2. Simulate Sensor Covariance + Test Data

In [3]:
# Sensor covariance from diverse simulated activity (stands in for resting-state EEG)
print('Simulating sensor covariance...')
config_cov = SimulationConfig(
    batch_size=500, n_sources=(1, 5), n_timepoints=50,
    snr_range=(0, 10), verbose=0, random_seed=42,
)
C_data = np.zeros((61, 61))
gen = SimulationGenerator(fwd_true, config_cov)
for _ in range(10):
    batch = next(gen.generate())
    for s in range(batch.x.shape[0]):
        x = batch.x[s]
        C_data += x @ x.T / x.shape[1]
C_data /= 5000

# Test data: sparse sources for localization visualization
print('Simulating test data...')
test_config = SimulationConfig(
    batch_size=1, n_sources=2, n_timepoints=200,
    snr_range=(10, 10), verbose=0, random_seed=7,
)
test_batch = next(SimulationGenerator(fwd_true, test_config).generate())
print('Done.')

Simulating sensor covariance...


/Users/lukas/projects/invert/invert-package/invert/util/source_adjacency.py:214: RuntimeWarning: 0.2% of original source space vertices have been omitted, tri-based adjacency will have holes.
Consider using distance-based adjacency or morphing data to all source space vertices.
  adjacency = mne.spatial_src_adjacency(src, verbose=verbose)


Simulating test data...
Done.


## 3. Prepare fsaverage Template for Optimization

In [4]:
print('Setting up fsaverage BEM (one-time K factorization)...')
bem_model = mne.make_bem_model(
    'fsaverage', ico=3, conductivity=(0.3, 0.006, 0.3),
    subjects_dir=subjects_dir, verbose=0,
)
bem, coeff, coeff_h = make_bem_solution_factored(bem_model)

# Store original inner skull geometry for co-deformation
skull_rr_orig = bem_model[2]['rr'].copy()
skull_nn_orig = bem_model[2]['nn'].copy()
skull_center = skull_rr_orig.mean(axis=0)

src_base = mne.setup_source_space(
    'fsaverage', spacing='ico3', surface='white',
    subjects_dir=subjects_dir, add_dist=False, verbose=0,
)
pos_base = np.vstack([
    src_base[0]['rr'][src_base[0]['vertno']].copy(),
    src_base[1]['rr'][src_base[1]['vertno']].copy(),
])
nn_base = np.vstack([
    src_base[0]['nn'][src_base[0]['vertno']].copy(),
    src_base[1]['nn'][src_base[1]['vertno']].copy(),
])
center = pos_base.mean(axis=0)
n_lh = len(src_base[0]['vertno'])
print(f'Template: {len(pos_base)} sources, center={center * 1000}')

Setting up fsaverage BEM (one-time K factorization)...
        outer skin  (642) -> outer skin  (642) ...
        outer skin  (642) -> outer skull (642) ...
        outer skin  (642) -> inner skull (642) ...
        outer skull (642) -> outer skin  (642) ...
        outer skull (642) -> outer skull (642) ...
        outer skull (642) -> inner skull (642) ...
        inner skull (642) -> outer skin  (642) ...
        inner skull (642) -> outer skull (642) ...
        inner skull (642) -> inner skull (642) ...
        inner skull (642) -> inner skull (642) ...
        Combining...
        Scaling...
Template: 1284 sources, center=[  0.24539509 -21.39095212  17.67677854]


In [5]:
def _apply_transform(pts, ctr, offset, stretch, rotation):
    """Apply rotation (degrees) + stretch + offset to points around center."""
    p = pts - ctr
    if np.any(rotation != 0):
        R = Rotation.from_euler('XYZ', rotation, degrees=True).as_matrix()
        p = p @ R.T
    return p * stretch + ctr + offset


def build_fwd(sigma_skull, sigma_scalp=0.3, offset=np.zeros(3),
              stretch=np.ones(3), rotation=np.zeros(3), co_deform=False):
    """Build forward model with global parameter transformation.

    Parameters
    ----------
    sigma_skull : float
        Skull conductivity (S/m).
    sigma_scalp : float
        Scalp conductivity (S/m).
    offset, stretch, rotation : array-like
        Geometric transform applied to sources (and inner skull if co_deform).
    co_deform : bool
        If True, also deform the inner skull BEM surface.
    """
    if co_deform:
        # Co-deform inner skull and recompute affected K blocks
        new_skull_rr = _apply_transform(skull_rr_orig, skull_center,
                                         offset, stretch, rotation)
        bem['surfs'][2]['rr'] = new_skull_rr
        _check_complete_surface(bem['surfs'][2])
        coeff_updated = partial_K_update(bem['surfs'], coeff, changed_idx=2)
        bem['surfs'][2]['rr'] = skull_rr_orig  # restore for next call
        rebuild_bem_from_coeff(bem, coeff_updated, coeff_h,
                               (sigma_scalp, sigma_skull, 0.3))
    else:
        rebuild_bem_from_coeff(bem, coeff, coeff_h,
                               (sigma_scalp, sigma_skull, 0.3))

    # Deform source positions and normals
    pos = _apply_transform(pos_base, center, offset, stretch, rotation)
    nn = nn_base.copy()
    if np.any(rotation != 0):
        R = Rotation.from_euler('XYZ', rotation, degrees=True).as_matrix()
        nn = nn @ R.T

    src = src_base.copy()
    for i, h in enumerate(src):
        sl = slice(0, n_lh) if i == 0 else slice(n_lh, None)
        h['rr'] = h['rr'].copy()
        h['rr'][h['vertno']] = pos[sl]
        h['nn'] = h['nn'].copy()
        h['nn'][h['vertno']] = nn[sl]
    fwd = mne.make_forward_solution(
        info, trans=trans_fs, src=src, bem=bem,
        eeg=True, mindist=0.0, verbose=0,
    )
    return mne.convert_forward_solution(
        fwd, surf_ori=True, force_fixed=True, verbose=0,
    )


def subspace_cost(G, k=10):
    """Subspace alignment: 1 - mean(SVD of eigenvector overlap)."""
    _, U_d = np.linalg.eigh(C_data)
    U_d = U_d[:, -k:]
    _, U_m = np.linalg.eigh(G @ G.T)
    U_m = U_m[:, -k:]
    S = np.linalg.svd(U_d.T @ U_m, compute_uv=False)
    return 1 - np.mean(S**2)

## 4. Optimize Global Parameters (Stepwise)

Three-stage optimization, saving intermediate results at each stage:
1. **σ only** — skull conductivity grid search
2. **σ + offset** — add brain position within skull (4 params)
3. **σ + offset + stretch** — add per-axis brain shape (7 params)

Each stage builds on the previous optimum. We save all intermediate
forward models and deformed subjects for comparative evaluation.

In [6]:
# Check cache for the final result
cached = load_params(C_data)
if cached is not None:
    print(f'Optimization loaded from cache (cost={cached["cost"]:.5f}, '
          f'co_deform={cached["co_deform"]})')

# --- Stage 1: σ grid search ---
print('\nStage 1: Skull conductivity grid search...')
sigmas = np.logspace(np.log10(0.002), np.log10(0.02), 15)
best_cost, best_sig = np.inf, None
for sig in sigmas:
    try:
        fwd = build_fwd(sigma_skull=sig)
        c = subspace_cost(fwd['sol']['data'])
        if c < best_cost:
            best_cost, best_sig = c, sig
    except Exception:
        pass

sig1, off1, stretch1 = best_sig, np.zeros(3), np.ones(3)
cost1 = best_cost
print(f'  σ = {sig1 * 1000:.1f} mS/m, cost = {cost1:.4f}')

Optimization loaded from cache (cost=0.04531, co_deform=True)

Stage 1: Skull conductivity grid search...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
  σ = 20.0 mS/m, cost = 0.2104


In [7]:
# --- Stage 2: σ + offset ---
print('\nStage 2: σ + offset (Powell)...')
n_eval = [0]

def joint_cost_4(params):
    sig = 10 ** params[0]
    off = np.array(params[1:4])
    n_eval[0] += 1
    if n_eval[0] % 50 == 0:
        print(f'  eval {n_eval[0]}...', flush=True)
    try:
        fwd = build_fwd(sigma_skull=sig, offset=off)
        return subspace_cost(fwd['sol']['data'])
    except Exception:
        return 1.0

x0 = [np.log10(sig1), 0, 0, 0]
res2 = sp_minimize(joint_cost_4, x0, method='Powell',
                   options={'maxiter': 300, 'ftol': 1e-6})
sig2, off2, stretch2 = 10 ** res2.x[0], res2.x[1:4], np.ones(3)
cost2 = res2.fun
print(f'  σ = {sig2 * 1000:.1f} mS/m')
print(f'  offset = [{off2[0]*1000:.1f}, {off2[1]*1000:.1f}, {off2[2]*1000:.1f}] mm')
print(f'  cost = {cost2:.4f} ({n_eval[0]} evals)')


Stage 2: σ + offset (Powell)...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
       

In [8]:
# --- Stage 3: σ + offset + stretch ---
print('\nStage 3: σ + offset + stretch (Powell)...')
n_eval[0] = 0

def full_cost_7(params):
    sig = 10 ** params[0]
    off = np.array(params[1:4])
    stretch = np.array(params[4:7])
    n_eval[0] += 1
    if n_eval[0] % 50 == 0:
        print(f'  eval {n_eval[0]}...', flush=True)
    try:
        fwd = build_fwd(sig, offset=off, stretch=stretch)
        return subspace_cost(fwd['sol']['data'])
    except Exception:
        return 1.0

x0_full = list(res2.x) + [1.0, 1.0, 1.0]
res3 = sp_minimize(full_cost_7, x0_full, method='Powell',
                   options={'maxiter': 500, 'ftol': 1e-6})
sig3 = 10 ** res3.x[0]
off3 = res3.x[1:4]
stretch3 = res3.x[4:7]
cost3 = res3.fun
print(f'  σ = {sig3 * 1000:.1f} mS/m')
print(f'  offset = [{off3[0]*1000:.1f}, {off3[1]*1000:.1f}, {off3[2]*1000:.1f}] mm')
print(f'  stretch = [{stretch3[0]:.3f}, {stretch3[1]:.3f}, {stretch3[2]:.3f}]')
print(f'  cost = {cost3:.4f} ({n_eval[0]} evals)')

# --- Summary of stepwise optimization ---
print('\n' + '=' * 60)
print(f'{"Stage":<30s} {"σ (mS/m)":>9s} {"Cost":>8s} {"Δ Cost":>8s}')
print('-' * 60)
vanilla_cost = subspace_cost(build_fwd(0.0042)['sol']['data'])
print(f'{"Vanilla (σ=4.2)":<30s} {4.2:9.1f} {vanilla_cost:8.4f} {"—":>8s}')
print(f'{"+ σ optimized":<30s} {sig1*1000:9.1f} {cost1:8.4f} {cost1-vanilla_cost:+8.4f}')
print(f'{"+ offset":<30s} {sig2*1000:9.1f} {cost2:8.4f} {cost2-cost1:+8.4f}')
print(f'{"+ stretch":<30s} {sig3*1000:9.1f} {cost3:8.4f} {cost3-cost2:+8.4f}')
print('=' * 60)
print('(Stage 4: co-deform + rotation in next cell)')


Stage 3: σ + offset + stretch (Powell)...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling...
        Combining...
        Scaling.

## 4b. Stage 4: Co-deform + Rotation (11 parameters) — Current Best

The best model jointly optimizes **11 parameters**: 2 ratio-constrained
conductivities + 3 offset + 3 stretch + 3 rotation. All geometric transforms
(offset, stretch, rotation) **co-deform** both source positions AND the inner
skull BEM surface, keeping skull-to-cortex distance physically consistent.

This was optimized via nested 2-stage Powell (geometry outer, σ sweep inner)
in `scripts/eval_rotation_codeform.py`. The parameters below are the cached
optimum — re-run that script to re-optimize from scratch.

**Why co-deform wins:** Source-only deformation creates unphysical geometry
(brain shifted but skull frozen). Co-deformation keeps the BEM consistent,
giving better spatial patterns (EMD) while preserving source coverage.

In [9]:
# --- Stage 4: Co-deform + rotation (winning model) ---
# Load from cache if available, otherwise use the known-good parameters
# from eval_rotation_codeform.py (Experiment 3).
cached = load_params(C_data)
if cached is not None and cached['co_deform']:
    sig4 = cached['sigma_skull']
    sig4_scalp = cached['sigma_scalp']
    off4 = cached['offset']
    stretch4 = cached['stretch']
    rot4 = cached['rotation']
    cost4 = cached['cost']
    print(f'Loaded from cache (cost={cost4:.5f})')
else:
    # Known-good parameters from nested Powell optimization
    sig4 = 0.0209         # 20.9 mS/m
    sig4_scalp = 0.210    # S/m
    off4 = np.array([-0.0003, 0.0069, 0.0099])   # meters
    stretch4 = np.array([1.005, 0.992, 0.985])
    rot4 = np.array([-1.9, -0.8, -2.3])           # degrees
    # Evaluate cost
    fwd_tmp = build_fwd(sig4, sigma_scalp=sig4_scalp, offset=off4,
                        stretch=stretch4, rotation=rot4, co_deform=True)
    cost4 = subspace_cost(fwd_tmp['sol']['data'])
    print(f'Using hardcoded params (cost={cost4:.5f})')

print(f'  σ_skull  = {sig4 * 1000:.1f} mS/m')
print(f'  σ_scalp  = {sig4_scalp * 1000:.0f} mS/m')
print(f'  offset   = [{off4[0]*1000:.1f}, {off4[1]*1000:.1f}, {off4[2]*1000:.1f}] mm')
print(f'  stretch  = [{stretch4[0]:.3f}, {stretch4[1]:.3f}, {stretch4[2]:.3f}]')
print(f'  rotation = [{rot4[0]:.1f}, {rot4[1]:.1f}, {rot4[2]:.1f}] deg')
print(f'  co_deform = True')

Loaded from cache (cost=0.04531)
  σ_skull  = 20.9 mS/m
  σ_scalp  = 210 mS/m
  offset   = [-0.3, 6.9, 9.9] mm
  stretch  = [1.005, 0.992, 0.985]
  rotation = [-1.9, -0.8, -2.3] deg
  co_deform = True


## 5. Build All Stepwise Forward Models & Deformed Subjects

Build a forward model and FreeSurfer subject for each optimization stage,
so we can evaluate and visualize each independently.

In [10]:
def create_deformed_subject(name, offset, stretch, rotation, center, subjects_dir):
    """Create a FreeSurfer subject with fsaverage surfaces deformed by offset + stretch + rotation."""
    fs_dir = _get_fsaverage_dir(verbose=0)
    subj_dir = os.path.join(subjects_dir, name, 'surf')
    os.makedirs(subj_dir, exist_ok=True)
    center_mm = center * 1000
    for hemi_name in ['lh', 'rh']:
        pos_full, faces = mne.read_surface(
            os.path.join(fs_dir, 'surf', f'{hemi_name}.white'))
        pos_deformed = _apply_transform(pos_full, center_mm,
                                         offset * 1000, stretch, rotation)
        mne.write_surface(os.path.join(subj_dir, f'{hemi_name}.white'),
                          pos_deformed.astype(np.float32), faces, overwrite=True)
        curv = os.path.join(fs_dir, 'surf', f'{hemi_name}.curv')
        if os.path.exists(curv):
            shutil.copy2(curv, os.path.join(subj_dir, f'{hemi_name}.curv'))
    return name


# Build all stages (including cached co-deform if available)
stages = [
    ('Vanilla (σ=4.2)',        dict(sigma_skull=0.0042)),
    ('+ σ optimized',          dict(sigma_skull=sig1)),
    ('+ σ + offset',           dict(sigma_skull=sig2, offset=off2)),
    ('+ σ + offset + stretch', dict(sigma_skull=sig3, offset=off3, stretch=stretch3)),
]

# Add cached co-deform + rotation model if available
cached = load_params(C_data)
if cached is not None and cached['co_deform']:
    stages.append((
        '+ co-deform + rotation (cached)',
        dict(sigma_skull=cached['sigma_skull'], sigma_scalp=cached['sigma_scalp'],
             offset=cached['offset'], stretch=cached['stretch'],
             rotation=cached['rotation'], co_deform=True),
    ))
    print(f'Loaded cached co-deform model: cost={cached["cost"]:.5f}')

fwds = {}
subj_names = {}
print('Building forward models for each stage...')
for label, kw in stages:
    fwd = build_fwd(**kw)
    fwds[label] = fwd
    n = fwd['sol']['data'].shape[1]

    # Create deformed subject for visualization
    off = kw.get('offset', np.zeros(3))
    stch = kw.get('stretch', np.ones(3))
    rot = kw.get('rotation', np.zeros(3))
    sname = label.replace(' ', '_').replace('(', '').replace(')', '').replace('=', '').replace('+', 'plus').strip('_')
    sname = f'fsavg_{sname}'
    create_deformed_subject(sname, off, stch, rot, center, subjects_dir)
    subj_names[label] = sname

    print(f'  {label:<40s}: {n} dipoles → subject "{sname}"')

print('Done.')

Loaded cached co-deform model: cost=0.04531
Building forward models for each stage...
        Combining...
        Scaling...
Overwriting existing file.
Overwriting existing file.
  Vanilla (σ=4.2)                         : 1284 dipoles → subject "fsavg_Vanilla_σ4.2"
        Combining...
        Scaling...
Overwriting existing file.
Overwriting existing file.
  + σ optimized                           : 1284 dipoles → subject "fsavg_plus_σ_optimized"
        Combining...
        Scaling...
Overwriting existing file.
Overwriting existing file.
  + σ + offset                            : 1196 dipoles → subject "fsavg_plus_σ_plus_offset"
        Combining...
        Scaling...
Overwriting existing file.
Overwriting existing file.
  + σ + offset + stretch                  : 1195 dipoles → subject "fsavg_plus_σ_plus_offset_plus_stretch"
        Combining...
        Scaling...
  + co-deform + rotation (cached)         : 1271 dipoles → subject "fsavg_plus_co-deform_plus_rotation_cached"
Done.


## 6. Single-Trial Visualization

Simulate one test trial and show: ground truth on sample, then each optimization
stage on its respective deformed fsaverage.

In [11]:
# Single test trial
test_config = SimulationConfig(
    batch_size=1, n_orders=(0, 3), n_sources=2, n_timepoints=200,
    snr_range=(100, 100), verbose=0, random_seed=None,
)
test_batch = next(SimulationGenerator(fwd_true, test_config).generate())

info_mne = mne.create_info(
    ch_names=fwd_true['info']['ch_names'], sfreq=250., ch_types='eeg',
)
evoked_test = mne.EvokedArray(test_batch.x[0], info_mne, tmin=0)

# Ground truth STC
verts_true = [fwd_true['src'][h]['vertno'] for h in range(2)]
stc_true = mne.SourceEstimate(
    test_batch.y[0], verts_true, tmin=0, tstep=1 / 250., subject='sample',
)
peak_time = stc_true.times[np.argmax(np.max(np.abs(stc_true.data), axis=0))]
print(f'Peak time: {peak_time:.3f}s')

Peak time: 0.788s


In [12]:
import tempfile

solver_name = 'subspace-sbl'
# solver_name = 'ESMV'

plot_kw = dict(
    surface='white', hemi='both', cortex='low_contrast',
    time_unit='s', size=(800, 400), smoothing_steps=10,
    initial_time=peak_time, verbose=0,
)

def show_brain(stc, subject, subjects_dir, title):
    brain = stc.plot(
        subjects_dir=subjects_dir, subject=subject,
        brain_kwargs={'title': title}, **plot_kw,
    )
    f = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    brain.save_image(f.name)
    
    # display(Image(filename=f.name))

# Ground truth
print('Ground truth (sample brain):')
show_brain(stc_true, 'sample', subjects_dir_sample, 'Ground Truth (sample)')

# Each optimization stage
for label, fwd_est in fwds.items():
    s = Solver(solver_name)
    s.make_inverse_operator(fwd_est, evoked_test, alpha='auto', verbose=0)
    stc_est = s.apply_inverse_operator(evoked_test)
    stc_est.subject = subj_names[label]

    print(f'\n{solver_name} — {label}:')
    show_brain(stc_est, subj_names[label], subjects_dir,
               f'{solver_name} — {label}')

Ground truth (sample brain):

subspace-sbl — Vanilla (σ=4.2):

subspace-sbl — + σ optimized:


/Users/lukas/projects/invert/invert-package/invert/util/source_adjacency.py:214: RuntimeWarning: 6.9% of original source space vertices have been omitted, tri-based adjacency will have holes.
Consider using distance-based adjacency or morphing data to all source space vertices.
  adjacency = mne.spatial_src_adjacency(src, verbose=verbose)



subspace-sbl — + σ + offset:


/Users/lukas/projects/invert/invert-package/invert/util/source_adjacency.py:214: RuntimeWarning: 6.9% of original source space vertices have been omitted, tri-based adjacency will have holes.
Consider using distance-based adjacency or morphing data to all source space vertices.
  adjacency = mne.spatial_src_adjacency(src, verbose=verbose)



subspace-sbl — + σ + offset + stretch:


/Users/lukas/projects/invert/invert-package/invert/util/source_adjacency.py:214: RuntimeWarning: 1.0% of original source space vertices have been omitted, tri-based adjacency will have holes.
Consider using distance-based adjacency or morphing data to all source space vertices.
  adjacency = mne.spatial_src_adjacency(src, verbose=verbose)



subspace-sbl — + co-deform + rotation (cached):


## 7. Quantitative Comparison: All Stages (Multi-Trial MLE + EMD)

Evaluate each optimization stage with 30 trials using cross-distance matrices.
This shows the **marginal contribution** of each parameter to actual source
reconstruction quality — not just subspace alignment cost.

In [13]:
import importlib.util, sys

# Direct import to bypass broken __init__.py chain (invert.models missing)
import invert as _invert_pkg
_eval_path = os.path.join(os.path.dirname(_invert_pkg.__file__), 'evaluate', 'evaluate.py')
_spec = importlib.util.spec_from_file_location(
    'invert.evaluate.evaluate', _eval_path, submodule_search_locations=[],
)
_eval_mod = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = _eval_mod
_spec.loader.exec_module(_eval_mod)
eval_emd = _eval_mod.eval_emd
eval_mean_localization_error = _eval_mod.eval_mean_localization_error

# Generate evaluation trials
n_trials = 30
eval_config = SimulationConfig(
    batch_size=n_trials, n_sources=2, n_timepoints=200,
    snr_range=(50, 50), verbose=0, random_seed=None,
)
eval_batch = next(SimulationGenerator(fwd_true, eval_config).generate())
print(f'{n_trials} evaluation trials generated')

# True source positions and adjacency
pos_sample = fwd_true['source_rr']
adj_true = mne.spatial_src_adjacency(fwd_true['src'], verbose=0)

30 evaluation trials generated


/var/folders/l1/86b52b5s5w770zbdp9qtgk5h0000gp/T/ipykernel_1105/3992569514.py:26: RuntimeWarning: 0.2% of original source space vertices have been omitted, tri-based adjacency will have holes.
Consider using distance-based adjacency or morphing data to all source space vertices.
  adj_true = mne.spatial_src_adjacency(fwd_true['src'], verbose=0)


In [14]:
def cross_subject_eval(fwd_est, solver_name):
    """Evaluate a forward model across multiple trials with MLE + EMD."""
    pos_est = fwd_est['source_rr']
    adj_est = mne.spatial_src_adjacency(fwd_est['src'], verbose=0)
    D_cross = cdist(pos_sample, pos_est)

    solver = Solver(solver_name)
    mles, emds = [], []
    for trial in range(n_trials):
        evoked = mne.EvokedArray(eval_batch.x[trial], info_mne, tmin=0)
        try:
            solver.make_inverse_operator(fwd_est, evoked, alpha='auto', verbose=0)
            stc = solver.apply_inverse_operator(evoked)

            y_true_avg = np.abs(eval_batch.y[trial]).mean(axis=1)
            y_pred_avg = np.abs(stc.data).mean(axis=1)

            mle = eval_mean_localization_error(
                y_true_avg, y_pred_avg,
                adj_true, adj_est, pos_sample, pos_est, D_cross,
            )
            emd_val = eval_emd(D_cross, y_true_avg, y_pred_avg)
            mles.append(mle * 1000)
            emds.append(emd_val * 1000)
        except Exception:
            pass
    return np.array(mles), np.array(emds)


eval_solver = 'subspace-sbl'
# eval_solver = 'ESMV'
print(f'Evaluating all stages with {eval_solver} ({n_trials} trials each)...')
print()

results = {}
for label, fwd_est in fwds.items():
    print(f'  {label}...')
    mles, emds = cross_subject_eval(fwd_est, eval_solver)
    results[label] = (mles, emds)

# Print results table
print()
print(f'{"Stage":<30s} {"MLE med":>8s} {"±std":>6s} {"EMD med":>8s} {"±std":>6s} {"n_src":>6s}')
print('-' * 68)
for label in fwds:
    mles, emds = results[label]
    n = fwds[label]['sol']['data'].shape[1]
    print(f'{label:<30s} {np.median(mles):7.1f}  {np.std(mles):5.1f} {np.median(emds):7.1f}  {np.std(emds):5.1f} {n:6d}')

# Improvement relative to vanilla
van_mle = np.median(results[stages[0][0]][0])
van_emd = np.median(results[stages[0][0]][1])
print()
print('Improvement vs vanilla:')
for label in list(fwds.keys())[1:]:
    mles, emds = results[label]
    print(f'  {label:<28s}  MLE: {(1 - np.median(mles)/van_mle)*100:+5.1f}%  EMD: {(1 - np.median(emds)/van_emd)*100:+5.1f}%')

Evaluating all stages with subspace-sbl (30 trials each)...

  Vanilla (σ=4.2)...
  + σ optimized...
  + σ + offset...


/var/folders/l1/86b52b5s5w770zbdp9qtgk5h0000gp/T/ipykernel_1105/593631963.py:4: RuntimeWarning: 6.9% of original source space vertices have been omitted, tri-based adjacency will have holes.
Consider using distance-based adjacency or morphing data to all source space vertices.
  adj_est = mne.spatial_src_adjacency(fwd_est['src'], verbose=0)


  + σ + offset + stretch...


/var/folders/l1/86b52b5s5w770zbdp9qtgk5h0000gp/T/ipykernel_1105/593631963.py:4: RuntimeWarning: 6.9% of original source space vertices have been omitted, tri-based adjacency will have holes.
Consider using distance-based adjacency or morphing data to all source space vertices.
  adj_est = mne.spatial_src_adjacency(fwd_est['src'], verbose=0)


  + co-deform + rotation (cached)...


/var/folders/l1/86b52b5s5w770zbdp9qtgk5h0000gp/T/ipykernel_1105/593631963.py:4: RuntimeWarning: 1.0% of original source space vertices have been omitted, tri-based adjacency will have holes.
Consider using distance-based adjacency or morphing data to all source space vertices.
  adj_est = mne.spatial_src_adjacency(fwd_est['src'], verbose=0)



Stage                           MLE med   ±std  EMD med   ±std  n_src
--------------------------------------------------------------------
Vanilla (σ=4.2)                   21.5   11.2    31.3   13.3   1284
+ σ optimized                     19.4    9.4    28.4   16.6   1284
+ σ + offset                      18.1    7.8    26.0   17.0   1196
+ σ + offset + stretch            18.5    8.6    25.8   17.6   1195
+ co-deform + rotation (cached)    18.7    8.6    26.3   16.3   1271

Improvement vs vanilla:
  + σ optimized                 MLE:  +9.7%  EMD:  +9.4%
  + σ + offset                  MLE: +15.8%  EMD: +17.0%
  + σ + offset + stretch        MLE: +13.9%  EMD: +17.5%
  + co-deform + rotation (cached)  MLE: +12.9%  EMD: +16.0%


## 8. Alignment Diagnostic

Nearest-neighbor distances between source meshes reveal the geometric floor
for localization error. Even a perfect solver can't localize better than the
NN distance, since the estimated source can only be placed at a fsaverage vertex.

In [ ]:
# Nearest-neighbor distances between source meshes (in head coordinates)
from scipy.spatial import cKDTree

print('=== Coordinate alignment diagnostic ===\n')

for label, fwd_est in fwds.items():
    pos_est = fwd_est['source_rr']
    tree_est = cKDTree(pos_est)
    nn_dist, _ = tree_est.query(pos_sample)

    c_est = pos_est.mean(axis=0) * 1000
    c_sample = pos_sample.mean(axis=0) * 1000
    centroid_dist = np.linalg.norm(c_sample - c_est)

    print(f'{label}:')
    print(f'  Centroid offset:  {centroid_dist:.1f} mm')
    print(f'  NN dist (mm):     mean={nn_dist.mean()*1000:.1f}  '
          f'median={np.median(nn_dist)*1000:.1f}  max={nn_dist.max()*1000:.1f}')
    print()

=== Coordinate alignment diagnostic ===

Vanilla (σ=4.2):
  Centroid offset:  17.7 mm
  NN dist (mm):     mean=6.5  median=5.7  max=19.9

+ σ optimized:
  Centroid offset:  17.7 mm
  NN dist (mm):     mean=6.5  median=5.7  max=19.9

+ σ + offset:
  Centroid offset:  5.8 mm
  NN dist (mm):     mean=5.3  median=5.1  max=14.8

+ σ + offset + stretch:
  Centroid offset:  5.7 mm
  NN dist (mm):     mean=5.5  median=5.2  max=16.8

+ co-deform + rotation (cached):
  Centroid offset:  6.6 mm
  NN dist (mm):     mean=5.2  median=5.0  max=13.2



: 